In [ ]:
import requests
import pandas as pd
import sqlite3

In [ ]:
server = "https://map.sepa.org.uk/server/rest/services/Open/Environmental_Monitoring/MapServer/1/query?f=json&where=(1%3D1)%20AND%20(1%3D1)&returnGeometry=true&spatialRel=esriSpatialRelIntersects&outFields=*&orderByFields=OBJECTID%20ASC&outSR=4326&resultOffset=0&resultRecordCount=100"


In [ ]:
data = requests.get(server, headers={"Accept": "application/json, text/javascript, */*; q=0.01"})

In [ ]:
data.json()

In [ ]:
df = pd.json_normalize(data.json()['features'])
df = df.rename(columns={
    "attributes.objectid": "id",
    "attributes.description": "name",
    "geometry.x": "lon",
    "geometry.y": "lat"
})\
.set_index("id")\
.drop(columns=[
    'attributes.class_description', 'attributes.bw_url',
    'attributes.year'
])
df['alternate_name'] = df['name']
locations = df[[
    'name', 'alternate_name', 'lat', 'lon'
]].copy()
locations

In [ ]:
locations

In [ ]:
db = sqlite3.connect("../dataset.sqlite3")
locations.to_sql("locations", db, if_exists="append")
